## Audio Feature Extraction

this notebook is for extracting audio features from tracks using librosa.

- **Update from Phase 1** : Due to spotify api changes, the `audio_analysis` endpoint can no longer be used. To extract the audio features from songs, we used `librosa` library to calculate the features (listed below)

#### Extracted Audio Features
| Feature | Description |
|---|---|
| `tempo` | Estimated BPM |
| `energy` | Mean RMS energy (loudness proxy) |
| `zero_crossing_rate` | Rate of sign changes (noisiness proxy) |
| `spectral_centroid` | Brightness / perceived pitch centre |
| `spectral_rolloff` | Frequency below which 85% of energy lies |
| `mfcc_1`, `mfcc_2` | First two Mel-frequency cepstral coefficients |
| `chroma_mean` | Mean chroma energy (harmonic content) |
| `chroma_std` | Std dev of chroma (harmonic variation) |

#### Main Functions and Utilities

- **Collecting** unique songs from weekly chart CSVs  
  - Download chart CSVs from the official Spotify weekly global charts (e.g., via `01_scrape_weekly_charts.ipynb`) and place them in this notebook’s working directory.  
  - `extract_date_from_filename` extracts the chart week (YYYY-MM-DD) from the filename.  
  - `load_unique_songs` reads all chart CSVs, keeps the top 50 per week, and deduplicates by Spotify URI (or track+artist) so each song’s audio features are computed only once.

- **Extracting** audio features via YouTube + librosa  
  - `download_and_analyze` searches YouTube for the track, downloads the first 60 seconds of audio, and uses librosa to compute 9 acoustic features (tempo, energy, zero-crossing rate, spectral centroid/rolloff, MFCC 1–2, chroma mean/std). Progress is checkpointed every 5 songs so runs can be resumed.

- **Retrying** failed downloads  
  - `retry_failed` re-runs extraction for songs where `tempo` is NaN (e.g., network errors or missing YouTube results). Checkpoints are saved every 10 songs.

- **Merging** features into the full chart dataset  
  - `merge_features` left-joins the extracted audio features onto every chart row and writes `../data/processed/processed_chart_tracks_audio_features.csv`.

In [33]:
import glob
import os
import re
import time

import librosa
import numpy as np
import pandas as pd
import yt_dlp

# Paths (uses cwd; no cookies)
SCRIPT_DIR = os.getcwd()
CHART_CSV_DIR = "/Users/mac/Downloads/dsw_hw2/chart_data"
FEATURES_FILE = os.path.join(SCRIPT_DIR, "unique_songs_audio_features.csv")

In [34]:
def download_and_analyze(track_name, artist_name):
    query = f"ytsearch1:{track_name} {artist_name} official audio"
    output_path = "temp_audio.%(ext)s"

    ydl_opts = {
        "format": "worstaudio/worst",  # avoids SABR streaming issue
        "outtmpl": output_path,
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "128",
        }],
        "quiet": True,
        "no_warnings": True,
        "noplaylist": True,
        "extractor_args": {
            "youtube": {
                "player_client": ["android"],  # use android client to bypass SABR
            }
        },
    }

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([query])

        mp3_path = "temp_audio.mp3"
        if not os.path.exists(mp3_path):
            return None

        # Analyze with librosa
        y, sr = librosa.load(mp3_path, duration=60)
        os.remove(mp3_path)

        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr)
        rms = librosa.feature.rms(y=y)
        zcr = librosa.feature.zero_crossing_rate(y)
        spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
        spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
        mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

        return {
            "tempo": float(np.atleast_1d(tempo)[0]),
            "energy": float(np.mean(rms)),
            "zero_crossing_rate": float(np.mean(zcr)),
            "spectral_centroid": float(np.mean(spectral_centroid)),
            "spectral_rolloff": float(np.mean(spectral_rolloff)),
            "mfcc_1": float(np.mean(mfcc[0])),
            "mfcc_2": float(np.mean(mfcc[1])),
            "chroma_mean": float(np.mean(chroma)),
            "chroma_std": float(np.std(chroma)),
        }
    except Exception as e:
        print(f"  Failed: {e}")
        if os.path.exists("temp_audio.mp3"):
            os.remove("temp_audio.mp3")
        return None

In [35]:
def extract_date_from_filename(path):
    """Pull a YYYY-MM-DD date from a filename (used as the chart week)."""
    match = re.search(r"(\d{4}-\d{2}-\d{2})", os.path.basename(path))
    return pd.to_datetime(match.group(1)) if match else pd.NaT


def load_unique_songs():
    """
    Read all chart CSVs (from downloads_temp or *.csv in cwd), keep the top 50 per week,
    and return (df_all, df_unique):
        df_all    — every chart row (top 50) with a 'date' column
        df_unique — one row per unique song (deduplicated by URI or track+artist)
    """
    files = sorted(glob.glob(os.path.join(CHART_CSV_DIR, "*.csv")))
    if not files:
        files = sorted(glob.glob(os.path.join(SCRIPT_DIR, "*.csv")))
    if not files:
        raise FileNotFoundError(f"No CSV files found in {CHART_CSV_DIR} or {SCRIPT_DIR}")

    print(f"Found {len(files)} chart CSV file(s)")

    frames = []
    for fp in files:
        try:
            df = pd.read_csv(fp)
            if "rank" in df.columns:
                df = df[df["rank"] <= 50]
            df["date"] = extract_date_from_filename(fp)
            frames.append(df)
        except Exception as e:
            print(f"  Skipping {fp}: {e}")

    df_all = pd.concat(frames, ignore_index=True)

    if "uri" in df_all.columns:
        df_unique = (
            df_all.dropna(subset=["uri"])
            .drop_duplicates(subset=["uri"])
            .reset_index(drop=True)
        )
    else:
        df_unique = (
            df_all.drop_duplicates(subset=["track_name", "artist_names"])
            .reset_index(drop=True)
        )

    print(f"  Total chart rows (top 50): {len(df_all)}")
    print(f"  Unique songs:              {len(df_unique)}")
    return df_all, df_unique

In [36]:
def extract_features():
    """
    For each unique song, download audio and compute features.
    Progress is checkpointed to FEATURES_FILE every 5 songs.
    No cookies used — relies on yt-dlp android client.
    """
    _, df_unique = load_unique_songs()

    processed_data = []
    processed_keys = set()

    if os.path.exists(FEATURES_FILE):
        existing = pd.read_csv(FEATURES_FILE)
        processed_data = existing.to_dict("records")
        for _, row in existing.iterrows():
            processed_keys.add(f"{row['track_name']}||{row['artist_names']}")
        print(f"  Resuming: {len(processed_keys)} songs already processed\n")

    remaining = [
        row for _, row in df_unique.iterrows()
        if f"{row['track_name']}||{row['artist_names']}" not in processed_keys
    ]

    if not remaining:
        print("All songs already processed — nothing to do.")
        return

    print(f"  {len(remaining)} songs remaining to process\n")

    for i, row in enumerate(remaining):
        t_name, a_name = row["track_name"], row["artist_names"]
        print(f"[{i + 1}/{len(remaining)}] {t_name} — {a_name}")

        feats = download_and_analyze(t_name, a_name)

        entry = {
            "uri": row.get("uri", None),
            "track_name": t_name,
            "artist_names": a_name,
        }
        if feats:
            entry.update(feats)
        else:
            entry["tempo"] = None

        processed_data.append(entry)

        if (i + 1) % 5 == 0 or (i + 1) == len(remaining):
            pd.DataFrame(processed_data).to_csv(FEATURES_FILE, index=False)
            print(f"    -- checkpoint saved ({len(processed_data)} total) --")

        time.sleep(1)

    print(f"\nFeature extraction complete. {len(processed_data)} songs saved.")

In [37]:
def retry_failed():
    """Re-attempt feature extraction for songs where tempo is NaN."""
    if not os.path.exists(FEATURES_FILE):
        print(f"{FEATURES_FILE} not found — run extraction first.")
        return

    df_feat = pd.read_csv(FEATURES_FILE)
    df_feat = (
        df_feat.drop_duplicates(subset=["track_name", "artist_names"], keep="first")
        .reset_index(drop=True)
    )

    failed = df_feat[df_feat["tempo"].isna()]
    if failed.empty:
        print("No failed songs to retry.")
        return

    print(f"Retrying {len(failed)} previously failed songs ...\n")

    recovered = 0
    for idx, (i, row) in enumerate(failed.iterrows()):
        t_name, a_name = row["track_name"], row["artist_names"]
        print(f"  [{idx + 1}/{len(failed)}] {t_name} — {a_name}")

        feats = download_and_analyze(t_name, a_name)
        if feats:
            for col, val in feats.items():
                df_feat.at[i, col] = val
            recovered += 1
            print("    Recovered!")

        if (idx + 1) % 10 == 0:
            df_feat.to_csv(FEATURES_FILE, index=False)
            print(f"    -- checkpoint saved ({recovered} recovered so far) --")

        time.sleep(1.5)

    df_feat.to_csv(FEATURES_FILE, index=False)
    has = df_feat["tempo"].notna().sum()
    still_missing = df_feat["tempo"].isna().sum()
    print(f"\nRetry done: recovered {recovered}/{len(failed)}")
    print(f"  With features: {has}  |  Still missing: {still_missing}")

In [38]:
def merge_features():
    """Left-join audio features onto every chart row and save the result."""
    FINAL_OUTPUT = os.path.join(SCRIPT_DIR, "../data/processed/processed_chart_tracks_audio_features.csv")

    if not os.path.exists(FEATURES_FILE):
        print(f"{FEATURES_FILE} not found — run extraction first.")
        return

    df_all, _ = load_unique_songs()

    df_feat = pd.read_csv(FEATURES_FILE)
    df_feat = (
        df_feat.drop_duplicates(subset=["track_name", "artist_names"], keep="first")
        .reset_index(drop=True)
    )

    has = df_feat["tempo"].notna().sum()
    missing = df_feat["tempo"].isna().sum()
    print(f"Features file: {len(df_feat)} songs ({has} with features, {missing} missing)")

    # Drop overlapping audio feature cols from df_all to avoid _x/_y suffixes
    feature_cols = [
        "tempo", "energy", "zero_crossing_rate", "spectral_centroid",
        "spectral_rolloff", "mfcc_1", "mfcc_2", "chroma_mean", "chroma_std",
    ]
    cols_to_drop = [c for c in feature_cols if c in df_all.columns]
    df_all_clean = df_all.drop(columns=cols_to_drop) if cols_to_drop else df_all

    features_to_merge = df_feat.drop(columns=["uri"], errors="ignore")
    final_df = df_all_clean.merge(
        features_to_merge, on=["track_name", "artist_names"], how="left"
    )

    matched = final_df["tempo"].notna().sum()
    print(f"Chart rows matched with features: {matched}/{len(final_df)}")

    final_df.to_csv(FINAL_OUTPUT, index=False)
    print(f"\nFinal dataset saved to {FINAL_OUTPUT}")

In [39]:
# 1. Collect — load chart CSVs and identify unique songs
# 2. Extract — download audio from YouTube (no cookies) and compute features
# 3. Retry  — re-attempt any failed songs
# 4. Merge  — join features back onto chart data

extract_features()
retry_failed()
merge_features()

Found 2 chart CSV file(s)
  Total chart rows (top 50): 100
  Unique songs:              54
  Resuming: 5 songs already processed

  49 songs remaining to process

[1/49] EoO — Bad Bunny
[2/49] The Fate of Ophelia — Taylor Swift                
[3/49] Man I Need — Olivia Dean                            
[4/49] VOY A LLeVARTE PA PR — Bad Bunny                    
[5/49] back to friends — sombr                           
    -- checkpoint saved (10 total) --                      
[6/49] LA CANCIÓN — J Balvin, Bad Bunny
[7/49] So Easy (To Fall In Love) — Olivia Dean             
[8/49] Ordinary — Alex Warren                              
[9/49] WHERE IS MY HUSBAND! — RAYE                         
[10/49] Opalite — Taylor Swift                             
    -- checkpoint saved (15 total) --                      
[11/49] Golden — HUNTR/X, EJAE, AUDREY NUNA, REI AMI, KPop Demon Hunters Cast
[12/49] Lush Life — Zara Larsson                           
[13/49] Qué Pasaría... — Rauw Alejandro,

* We collected five years of Spotify Global Chart data. This notebook shows only part of the data processing. For efficiency, we processed different subsets of the data separately to extract audio features (using this notebook), and then combined the results into the final dataset.


In [15]:
# Optional: re-run only retry + merge (skip full extract)
# retry_failed()
# merge_features()  

In [40]:
# Inspect csv output
final_df = pd.read_csv("../data/processed/processed_chart_tracks_audio_features.csv")
print(f"Done! {len(final_df)} rows. Columns: {list(final_df.columns)}")
final_df.head()

Done! 100 rows. Columns: ['rank', 'uri', 'artist_names', 'track_name', 'source', 'peak_rank', 'previous_rank', 'weeks_on_chart', 'streams', 'date', 'tempo', 'energy', 'zero_crossing_rate', 'spectral_centroid', 'spectral_rolloff', 'mfcc_1', 'mfcc_2', 'chroma_mean', 'chroma_std']


,rank,uri,artist_names,track_name,source,peak_rank,previous_rank,weeks_on_chart,streams,date,tempo,energy,zero_crossing_rate,spectral_centroid,spectral_rolloff,mfcc_1,mfcc_2,chroma_mean,chroma_std
0,1,spotify:track:3sK8wGT43QFpWrvNQsrQya,Bad Bunny,DtMF,Rimas Entertainment LLC.,1,5,58,80735628,2026-02-12,112.347147,0.147263,0.073504,1705.868348,3843.705924,-180.743683,122.638329,0.316952,0.306934
1,2,spotify:track:2lTm559tuIvatlT1u0JYG2,Bad Bunny,BAILE INoLVIDABLE,Rimas Entertainment LLC.,2,11,58,55193866,2026-02-12,172.265625,0.209944,0.063718,1927.437881,4397.244244,-103.004150,104.519508,0.373859,0.296914
2,3,spotify:track:5TFD2bmFKGhoCRbX61nXY5,Bad Bunny,NUEVAYoL,Rimas Entertainment LLC.,3,21,58,52228763,2026-02-12,117.453835,0.178102,0.089930,2270.121682,4786.825221,-105.814262,83.794174,0.413171,0.308954
3,4,spotify:track:6J5kc12BW5HuP3d7C3vvx8,Bad Bunny,EoO,Rimas Entertainment LLC.,4,27,58,41160736,2026-02-12,103.359375,0.192688,0.117968,2678.292530,5521.079115,-77.810104,74.532661,0.440456,0.307612
4,5,spotify:track:3qhlB30KknSejmIvZZLjOD,Djo,End of Beginning,Djo,1,1,104,38834551,2026-02-12,161.499023,0.212295,0.062188,1943.919670,4481.331235,-103.818100,99.461784,0.435699,0.292676


### Description of the limitations of, or issues you ran into, with the data that you are using.
##### Limitations and Issues

- **YouTube as audio source**: Audio is taken from YouTube search results (`ytsearch1`). The first result is not always the official track; it can be a cover, remix, or unofficial upload, so features may reflect the wrong version.
- **Audio format**: Downloads use `worstaudio/worst` to reduce SABR-related issues, which can lower audio quality compared to higher-bitrate sources.

##### Extraction

- **First 60 seconds only**: Features are computed on the first 60 seconds of audio, which may not represent the full song (e.g., tempo changes, instrumental sections).

##### Chart Data

- **Directory-based loading**: Chart CSVs are loaded from a directory via glob. If that directory contains non-chart CSVs (e.g. Airbnb, sports stats), they are concatenated and can distort the dataset or cause column clashes.
- **Filename assumptions**: Dates are parsed from filenames with `YYYY-MM-DD`. Non-standard filenames may result in missing or incorrect `date` values.

##### Feature Quality

- **Tempo**: Librosa’s tempo estimation can be unreliable for non-4/4 time signatures, mixed tempos, or long intros.
- **Feature stability**: Features are derived from one YouTube version. Different releases, remasters, or uploads of the same track can change the feature values.


### Description of Analysis-ready DataFrame: `../data/processed/processed_chart_tracks_audio_features.csv`

raw data sourced from : https://charts.spotify.com/charts/view/regional-global-weekly/2026-02-12

##### Rows
- **Date range** : 2021-01-07 to 2026-02-19
- **One row per chart entry** — each row is a song's appearance in one week's top-50 Spotify global chart.
- **Same song can appear across multiple weeks** — if a track stays on the chart for multiple weeks, it will have multiple rows (one per week).
- **Row count** = (number of chart weeks) × (up to 50 songs per week). Unique songs are deduplicated at extraction time (by URI or track+artist), but the merged dataset keeps every chart occurrence.

##### Columns

| Column | Source | Description |
|--------|--------|-------------|
| `rank` | Chart | Position (1–50) for that week |
| `uri` | Chart | Spotify track URI |
| `artist_names` | Chart | Artist name(s) |
| `track_name` | Chart | Track name |
| `source` | Chart | Record label/source |
| `peak_rank` | Chart | Best rank ever reached |
| `previous_rank` | Chart | Rank in the previous week |
| `weeks_on_chart` | Chart | Total weeks on the chart |
| `streams` | Chart | Stream count for that week |
| `date` | Derived | Chart week (YYYY-MM-DD, parsed from filename) |
| `tempo` | Extracted | BPM (from first 60 seconds of YouTube audio) |
| `energy` | Extracted | Mean RMS energy (loudness proxy) |
| `zero_crossing_rate` | Extracted | Rate of sign changes (noisiness proxy) |
| `spectral_centroid` | Extracted | Brightness / perceived pitch centre (Hz) |
| `spectral_rolloff` | Extracted | Frequency below which 85% of energy lies (Hz) |
| `mfcc_1`, `mfcc_2` | Extracted | First two Mel-frequency cepstral coefficients (timbre) |
| `chroma_mean` | Extracted | Mean chroma energy (harmonic content) |
| `chroma_std` | Extracted | Standard deviation of chroma (harmonic variation) |

##### Notes

- **Missing values**: Audio feature columns (`tempo` through `chroma_std`) may be `NaN` if YouTube download or extraction failed.
- **Deduplication**: Audio features are computed once per unique song and reused across all chart rows for that track.

### What specific questions do you have for your TA reviewer regarding your project thus far?

- For tracks where YouTube download or librosa extraction fails (e.g., no official audio, region block), should we leave those rows as NaN, drop them, or use a retry/fallback strategy—and is there a preference for how we describe this in the report?